In [0]:
## import biblioteca

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType,
    DateType,
    TimestampType
)

In [0]:
CATALOG = "workspace"
SCHEMA = "desafio_sicredi"

BRONZE_CLIENTES = f"{CATALOG}.{SCHEMA}.bronze_clientes"
SILVER_CLIENTES = f"{CATALOG}.{SCHEMA}.silver_clientes"
QUARENTENA_CLIENTES = f"{CATALOG}.{SCHEMA}.quarentena_clientes"

In [0]:
##ler_dados_bronze

df_clientes = spark.table(BRONZE_CLIENTES)

display(df_clientes)

id_cliente,nome,cpf,data_nascimento,cidade,estado,data_atualizacao,arquivo_origem,data_referencia,timestamp_ingestao,hash_linha
1,Ana Silva,52998224725,1990-05-10,Recife,PE,2026-10-13 08:00:00,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-2888250397107858912-8f925ce1-2208-4526-bc33-e01229810444-160-1-c000.csv,null,2026-10-15T13:47:57.529Z,fd1ce2688be3c72f4b1c10c5ac7dede6509daf302a560e6e592b1c3341053828
1,Ana Maria Silva,52998224725,1990-05-10,Recife,PE,2026-10-14 09:00:00,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-2888250397107858912-8f925ce1-2208-4526-bc33-e01229810444-160-1-c000.csv,null,2026-10-15T13:47:57.529Z,fc817641af95b624914b9a674db7e70d15ae0761720ff5cbd7cc4ba5312482a4
2,Carlos Souza,11111111111,1985-03-20,Olinda,PE,2026-10-13 10:00:00,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-2888250397107858912-8f925ce1-2208-4526-bc33-e01229810444-160-1-c000.csv,null,2026-10-15T13:47:57.529Z,566b60ee7891af04015b37faf867fe01acaa5ef1a695eeac2beb555180eb897b
3,Mariana Costa,16899535009,1995-07-15,Paulista,PE,2026-10-15 11:00:00,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-2888250397107858912-8f925ce1-2208-4526-bc33-e01229810444-160-1-c000.csv,null,2026-10-15T13:47:57.529Z,440b41e08101aed6e804e9be15f7cae395a06d5d06549fdbdb111cd47b77664a
1,Ana Silva,52998224725,1990-05-10,Recife,PE,2026-10-13 08:00:00,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-8433423682370044370-ef0b53ed-2782-416a-8a0d-0531f07063b0-140-1-c000.csv,null,2026-10-15T17:09:21.796Z,fd1ce2688be3c72f4b1c10c5ac7dede6509daf302a560e6e592b1c3341053828
1,Ana Maria Silva,52998224725,1990-05-10,Recife,PE,2026-10-14 09:00:00,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-8433423682370044370-ef0b53ed-2782-416a-8a0d-0531f07063b0-140-1-c000.csv,null,2026-10-15T17:09:21.796Z,fc817641af95b624914b9a674db7e70d15ae0761720ff5cbd7cc4ba5312482a4
2,Carlos Souza,11111111111,1985-03-20,Olinda,PE,2026-10-13 10:00:00,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-8433423682370044370-ef0b53ed-2782-416a-8a0d-0531f07063b0-140-1-c000.csv,null,2026-10-15T17:09:21.796Z,566b60ee7891af04015b37faf867fe01acaa5ef1a695eeac2beb555180eb897b
3,Mariana Costa,16899535009,1995-07-15,Paulista,PE,2026-10-15 11:00:00,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-8433423682370044370-ef0b53ed-2782-416a-8a0d-0531f07063b0-140-1-c000.csv,null,2026-10-15T17:09:21.796Z,440b41e08101aed6e804e9be15f7cae395a06d5d06549fdbdb111cd47b77664a


In [0]:
##conversao_dados

df_clientes_tipados = (
    df_clientes
    .withColumn(
        "id_cliente",
        F.expr("try_cast(id_cliente AS INT)")
    )
    .withColumn(
        "data_nascimento",
        F.expr("try_cast(data_nascimento AS DATE)")
    )
    .withColumn(
        "data_atualizacao",
        F.expr("try_cast(data_atualizacao AS TIMESTAMP)")
    )
)

df_clientes_tipados.printSchema()

root
 |-- id_cliente: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- cpf: string (nullable = true)
 |-- data_nascimento: date (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- data_atualizacao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)
 |-- data_referencia: void (nullable = true)
 |-- timestamp_ingestao: timestamp (nullable = true)
 |-- hash_linha: string (nullable = true)



In [0]:
##funcao_valida_cpf

def validar_cpf(cpf):
    if cpf is None:
        return False

    cpf = "".join(c for c in str(cpf) if c.isdigit())

    if len(cpf) != 11:
        return False

    if cpf == cpf[0] * 11:
        return False

    for tamanho in (9, 10):
        soma = sum(
            int(cpf[i]) * (tamanho + 1 - i)
            for i in range(tamanho)
        )

        resto = (soma * 10) % 11
        digito = 0 if resto == 10 else resto

        if digito != int(cpf[tamanho]):
            return False

    return True

In [0]:
from pyspark.sql.types import BooleanType

validar_cpf_udf = F.udf(validar_cpf, BooleanType())

In [0]:
df_clientes_validados = (
    df_clientes_tipados
    .withColumn(
        "cpf_valido",
        validar_cpf_udf(F.col("cpf"))
    )
)

display(
    df_clientes_validados.select(
        "id_cliente",
        "nome",
        "cpf",
        "cpf_valido"
    )
)

id_cliente,nome,cpf,cpf_valido
1,Ana Silva,52998224725,true
1,Ana Maria Silva,52998224725,true
2,Carlos Souza,11111111111,false
3,Mariana Costa,16899535009,true
1,Ana Silva,52998224725,true
1,Ana Maria Silva,52998224725,true
2,Carlos Souza,11111111111,false
3,Mariana Costa,16899535009,true


In [0]:
janela = (
    Window
    .partitionBy("id_cliente")
    .orderBy(
        F.col("data_atualizacao").desc_nulls_last(),
        F.col("timestamp_ingestao").desc_nulls_last(),
        F.col("hash_linha").desc()
    )
)

df_clientes_dedup = (
    df_clientes_validados
    .withColumn(
        "numero_linha",
        F.row_number().over(janela)
    )
    .filter(F.col("numero_linha") == 1)
    .drop("numero_linha")
)

display(df_clientes_dedup)

id_cliente,nome,cpf,data_nascimento,cidade,estado,data_atualizacao,arquivo_origem,data_referencia,timestamp_ingestao,hash_linha,cpf_valido
1,Ana Maria Silva,52998224725,1990-05-10,Recife,PE,2026-10-14T09:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-8433423682370044370-ef0b53ed-2782-416a-8a0d-0531f07063b0-140-1-c000.csv,null,2026-10-15T17:09:21.796Z,fc817641af95b624914b9a674db7e70d15ae0761720ff5cbd7cc4ba5312482a4,true
2,Carlos Souza,11111111111,1985-03-20,Olinda,PE,2026-10-13T10:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-8433423682370044370-ef0b53ed-2782-416a-8a0d-0531f07063b0-140-1-c000.csv,null,2026-10-15T17:09:21.796Z,566b60ee7891af04015b37faf867fe01acaa5ef1a695eeac2beb555180eb897b,false
3,Mariana Costa,16899535009,1995-07-15,Paulista,PE,2026-10-15T11:00:00.000Z,dbfs:/Volumes/workspace/desafio_sicredi/arquivos_entrada/clientes/part-00000-tid-8433423682370044370-ef0b53ed-2782-416a-8a0d-0531f07063b0-140-1-c000.csv,null,2026-10-15T17:09:21.796Z,440b41e08101aed6e804e9be15f7cae395a06d5d06549fdbdb111cd47b77664a,true


In [0]:
##registro_invalido

df_clientes_qualidade = (
    df_clientes_dedup
    .withColumn(
        "motivo_erro",
        F.when(
            F.col("id_cliente").isNull(),
            F.lit("ID_CLIENTE_INVALIDO")
        )
        .when(
            F.col("nome").isNull() |
            (F.trim(F.col("nome")) == ""),
            F.lit("NOME_OBRIGATORIO")
        )
        .when(
            ~F.coalesce(F.col("cpf_valido"), F.lit(False)),
            F.lit("CPF_INVALIDO")
        )
        .when(
            F.col("data_nascimento").isNull(),
            F.lit("DATA_NASCIMENTO_INVALIDA")
        )
        .when(
            F.col("data_atualizacao").isNull(),
            F.lit("DATA_ATUALIZACAO_INVALIDA")
        )
    )
)

In [0]:
##funcao_registros_validos_quarentena

df_clientes_validos = (
    df_clientes_qualidade
    .filter(F.col("motivo_erro").isNull())
    .drop("motivo_erro", "cpf_valido")
)

df_clientes_quarentena = (
    df_clientes_qualidade
    .filter(F.col("motivo_erro").isNotNull())
)

In [0]:
print("Clientes válidos:", df_clientes_validos.count())
print("Clientes em quarentena:", df_clientes_quarentena.count())

Clientes válidos: 2
Clientes em quarentena: 1


In [0]:
##insercao_tabela_delta

(
    df_clientes_validos.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_CLIENTES)
)

(
    df_clientes_quarentena.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(QUARENTENA_CLIENTES)
)

In [0]:
%sql
SELECT
    id_cliente,
    nome,
    cpf,
    cidade,
    estado
FROM workspace.desafio_sicredi.silver_clientes
ORDER BY id_cliente;

id_cliente,nome,cpf,cidade,estado
1,Ana Maria Silva,52998224725,Recife,PE
3,Mariana Costa,16899535009,Paulista,PE


In [0]:
%sql
SELECT
    id_cliente,
    nome,
    cpf,
    motivo_erro
FROM workspace.desafio_sicredi.quarentena_clientes;

id_cliente,nome,cpf,motivo_erro
2,Carlos Souza,11111111111,CPF_INVALIDO
